# HealthPilot-QA — Fine-tuning OpenBioLLM-8B (annotated)

**What this notebook does:** fine-tunes a medical LLM (`aaditya/Llama3-OpenBioLLM-8B`)
into **HealthPilot** — a clinical *support* assistant that reads a patient's symptoms
and history and returns organized, empathetic, **severity-tiered** guidance.

**Method:** QLoRA (4-bit base + small LoRA adapters) via **Unsloth**, so an 8B model
trains on a single Kaggle **T4 (16GB)**. Loss is masked to the **assistant turn only**.

**Pipeline context:** this model is the **response node** in a larger agent. Upstream
nodes (imaging model, EHR/medication model, encyclopedia RAG, a reasoning model that
compiles a report) do the heavy clinical analysis; HealthPilot communicates it.

> **Safety note (read once):** the model imitates the *style* of doctor answers and
> is **not** a reliable triage authority. We observed it under-triage a heart attack.
> Any "this is urgent / book / call 999" decision must come from a **deterministic rule
> layer**, never from the model alone. This is a research/prototype artifact — not
> validated for real clinical use.

---
## 0 · Environment & GPU check

In [ ]:
# ── Check the GPU ───────────────────────────────────────────────────────────
# Prints the NVIDIA driver + which GPU(s) Kaggle gave you (T4 / P100 / 2×T4).
# Confirms CUDA is visible BEFORE we waste time installing/loading anything.
# On Kaggle free tier you typically get a single T4 (16GB) or 2×T4.

!nvidia-smi

In [ ]:
# ── Disable HF 'Xet' fast-transfer backend ──────────────────────────────────
# Xet is HuggingFace's newer upload/download path. On Kaggle it has a version
# mismatch bug that crashes pushes/downloads with a confusing TypeError.
# Setting this to '1' forces the classic, reliable LFS path instead.
# MUST be set BEFORE huggingface_hub is first imported to take effect.

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

---
## 1 · Install dependencies
Torch (CUDA-matched) first, then the training stack, with **Unsloth last** so it can
pin compatible versions.

In [ ]:
# ── Install a CUDA-matched PyTorch ──────────────────────────────────────────
# Pulls torch built for CUDA 12.6. Matching torch's CUDA build to the runtime
# avoids 'CUDA capability'/driver mismatch errors later with bitsandbytes &
# flash/xformers kernels.

%pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
# ── Install the fine-tuning stack ───────────────────────────────────────────
# transformers/datasets/accelerate/peft/trl  → core HF training tooling
# bitsandbytes                                → 4-bit (QLoRA) quantization
# wandb                                       → live loss/metric dashboards
# hf_transfer / huggingface_hub               → model + dataset I/O with the Hub
# triton / xformers                           → fast attention/kernels
# unsloth                                     → the optimization layer that makes
#                                               an 8B trainable on a single T4
# NOTE: 'unsloth' is installed LAST so it can pin compatible versions of the rest.

%pip install -U transformers
%pip install -U datasets
%pip install -U accelerate
%pip install -U peft
%pip install -U trl
%pip install -U bitsandbytes
%pip install -U wandb
%pip install hf_transfer
%pip install -U huggingface_hub
%pip install triton
%pip install xformers
%pip install unsloth

In [ ]:
# Import torch on its own first — if the CUDA build is wrong, this is where
# you find out (rather than 10 cells later mid-training).

import torch

In [ ]:
# Quick sanity print of the installed torch version string.

torch.__version__

---
## 2 · Secrets, imports & experiment tracking
Pull tokens from Kaggle Secrets, import the training stack, and log into HF + wandb
**non-interactively** (so headless "Save & Run All" commits don't hang on a prompt).

In [ ]:
# ── Kaggle Secrets ──────────────────────────────────────────────────────────
# UserSecretsClient reads secrets you added under Add-ons → Secrets.
# We use it to pull HF_TOKEN and WANDB_API_KEY WITHOUT hard-coding them in the
# notebook (never paste raw tokens into a notebook you might share/commit).

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [ ]:
# ── Unsloth imports ─────────────────────────────────────────────────────────
# FastLanguageModel       → Unsloth's drop-in loader + LoRA wrapper (memory-optimized)
# train_on_responses_only → masks loss so the model only learns the ASSISTANT turn,
#                           not the system prompt or the user's question
# get_chat_template       → attaches the correct Llama-3.1 chat template to the tokenizer
#                           (the base repo's tokenizer ships without one)

import unsloth
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only, get_chat_template

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# ── Remaining imports ───────────────────────────────────────────────────────
# NOTE: peft's LoraConfig/get_peft_model are imported here but with Unsloth you
# should use FastLanguageModel.get_peft_model instead (see the LoRA cell below).
# These peft imports are effectively unused in the Unsloth path — harmless, but
# don't mix peft.get_peft_model with the Unsloth-wrapped model.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from peft import (
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)

import os
import torch
import wandb
from datasets import load_dataset, concatenate_datasets
from trl import SFTTrainer, SFTConfig
import bitsandbytes as bnb

In [ ]:
# Names the wandb run after this notebook (cosmetic; shows up in the dashboard).

os.environ["WANDB_NOTEBOOK_NAME"] = "content/healthPilot_QA.ipynb"

In [ ]:
# ── Hugging Face CLI login ──────────────────────────────────────────────────
# Interactive `hf auth login` will PROMPT for a token — fine when running cells
# by hand, but in a headless 'Save & Run All' commit there's no one to type it.
# For batch runs prefer: login non-interactively with the secret, e.g.
#   from huggingface_hub import login; login(token=user_secrets.get_secret('HF_TOKEN'))

!hf auth login

In [ ]:
# ── wandb login ─────────────────────────────────────────────────────────────
# Logs into Weights & Biases using the Kaggle secret so batch runs don't prompt.
# (Pass the key as `key=...`; non-interactive, no paste required.)

wandb.login(user_secrets.get_secret("WANDB_API_KEY"))

In [ ]:
# ── Start a wandb run ───────────────────────────────────────────────────────
# Opens the experiment-tracking run. All training loss/eval-loss curves stream
# here so you can monitor an overnight run from your phone.
# NOTE: project name still says 'BioClinical ModernBERT' — a leftover from an
# earlier plan. The model is actually OpenBioLLM-8B. Rename for clarity if you like.

run = wandb.init(
    project='Fine-tune BioClinical ModernBERT as healthPilot-QA',
    job_type="training",
    anonymous="allow"
)

---
## 3 · Identifiers & base-model download
Define the base model / output / dataset names, then download the base model **to a
local path** (sidesteps this repo's safetensors/discussions-403 quirk).

In [ ]:
# ── Key identifiers ─────────────────────────────────────────────────────────
# model_id     : the base model we fine-tune (OpenBioLLM-8B, a Llama-3.x medical model)
# new_model    : output repo/dir name for the fine-tuned result
# dataset_name : the source QA dataset on the Hub
# NOTE: 'new_model' still carries the old 'BioClinical-ModernBERT' name — purely
# a label, but worth renaming to something like 'healthpilot-openbiollm' to avoid
# confusion, since it is NOT a ModernBERT model.

model_id = "aaditya/Llama3-OpenBioLLM-8B"
new_model = "BioClinical-ModernBERT-large-HealthPilot-QA"
dataset_name = "Malikeh1375/medical-question-answering-datasets"

In [ ]:
# ── Download the base model to local disk ───────────────────────────────────
# snapshot_download pulls the full repo to the runtime cache and returns its path.
# WHY: loading FROM A LOCAL PATH avoids the 'discussions disabled (403)' / safetensors
# auto-conversion error that this repo triggers when loaded by name. Local files =
# no remote discussions lookup, no conversion thread.

from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="aaditya/Llama3-OpenBioLLM-8B",
    token=user_secrets.get_secret("HF_TOKEN"),
)

In [ ]:
# Show the local path snapshot_download returned (used as model_name below).

path

---
## 4 · Load the base model (4-bit) + tokenizer
Load OpenBioLLM in 4-bit with Unsloth and sanity-check the tokenizer and device.

In [ ]:
# ── Loading hyperparameters ─────────────────────────────────────────────────
# max_seq_length : max tokens per example the model will handle
# dtype=None     : let Unsloth auto-pick (bf16 if supported, else fp16)
# load_in_4bit   : QLoRA — load weights in 4-bit to fit the 8B on a 16GB T4

max_seq_length = 2048
dtype = None
load_in_4bit = True

In [ ]:
# ── Load the base model + tokenizer (Unsloth) ───────────────────────────────
# FastLanguageModel.from_pretrained returns BOTH the 4-bit model and its tokenizer,
# already patched for memory-efficient training.
# use_safetensors=False : this repo ships .bin weights; this skips the safetensors
#                         auto-conversion attempt that 403s on the disabled discussions API.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=path,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=user_secrets.get_secret("HF_TOKEN"),
    use_safetensors=False
)

In [ ]:
# ── Inspect tokenizer special tokens ────────────────────────────────────────
# Llama-3 ships with NO pad token by default. Unsloth assigns a dedicated reserved
# token as pad (better than reusing EOS). Expected healthy output:
#   pad_token: <|reserved_special_token_250|>  (a DEDICATED pad, not eos)
#   eos_token: <|end_of_text|>
#   padding_side: left   (Unsloth's intentional default — leave it)

print("pad_token:", tokenizer.pad_token)
print("pad_token_id:", tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token)
print("padding_side:", tokenizer.padding_side)

In [ ]:
# ── Confirm the model actually loaded ───────────────────────────────────────
# If the 403/safetensors warning printed in a background thread, this verifies the
# main load still succeeded. Prints the class, architecture, and which device the
# weights are on (cuda:0). If this runs without error, you're good.

print(type(model))
print(model.config.model_type)
print(next(model.parameters()).device)

---
## 5 · Data: load, format, split
Use the two **ChatDoctor** configs (real symptom→advice Q&A), wrap every example in
the HealthPilot system prompt + chat template, hold out an eval set, and subsample
the train set to **25k** for a session-sized run.

In [ ]:
# ── Load + combine the training data ────────────────────────────────────────
# We use ONLY the two ChatDoctor configs (real patient→doctor Q&A), NOT 'all-processed'.
# WHY: all-processed mixes in exam MCQs, literature summarization, and flashcards,
# which would teach the model the wrong behavior. ChatDoctor = symptom→advice, which
# matches the task. healthcaremagic (~112k) + icliniq (~7k), shuffled together.

a = load_dataset("malikeh1375/medical-question-answering-datasets", "chatdoctor_healthcaremagic")["train"]
b = load_dataset("malikeh1375/medical-question-answering-datasets", "chatdoctor_icliniq")["train"]
dataset = concatenate_datasets([a, b]).shuffle(seed=42)

In [3]:
# ── System prompt + chat formatting function ────────────────────────────────
# SYSTEM_INSTRUCTION: the persona/behavior baked into EVERY training example so the
#   model consistently learns the HealthPilot voice + severity-tiered triage.
# format_chat_template(batch):
#   - builds the patient turn from `instruction` (+ `input` when present)
#   - assembles a [system, user, assistant] message list
#   - renders it with the tokenizer's chat template into one 'text' string per row
#   NOTE: no add_generation_prompt here — the assistant ANSWER is included because
#   this text is for TRAINING (the model learns to produce that answer).

SYSTEM_INSTRUCTION = """You are 'HealthPilot', a clinical support assistant that helps patients \
understand their symptoms and decide what to do next. You are not a physician and you do not give a definitive \
diagnosis; you provide careful, well-reasoned guidance and a clear recommendation for next steps.

You are given a patient's described symptoms and relevant medical history. For each case you must:
1. Briefly restate your understanding of the patient's situation in plain, reassuring language.
2. Explain the most relevant considerations for their symptoms, without overstating certainty.
3. Recommend clear next steps, calibrated to severity:
   - Mild / self-limiting: sensible self-care, plus the specific warning signs that should prompt escalation.
   - Moderate: advise being seen by an appropriate clinician soon.
   - Severe or red-flag symptoms (e.g. chest pain, trouble breathing, stroke signs, severe bleeding, \
fainting, signs of sepsis): state clearly that this needs immediate medical attention and recommend \
urgent/emergency care.
4. When uncertain, err on the side of caution and recommend professional evaluation.

Always respond in an organized, professional, and empathetic tone. Use a clear structure. Never dismiss \
potentially serious symptoms. Always close by reminding the patient that this guidance does not replace \
assessment by a qualified healthcare professional."""

def format_chat_template(batch):
    instructions = batch["instruction"]
    inputs = batch.get("input", [""] * len(instructions))
    outputs = batch["output"]

    texts = []
    for i in range(len(instructions)):
        # Build the patient turn from instruction + input (when input is present)
        patient = instructions[i]
        if inputs[i] and inputs[i].strip():
            patient = f"{instructions[i]}\n\n{inputs[i]}".strip()

        messages = [
            {"role": "system", "content": SYSTEM_INSTRUCTION},
            {"role": "user", "content": patient},
            {"role": "assistant", "content": outputs[i]},
        ]
        # NOTE: no add_generation_prompt — the assistant turn is already included for training
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)

    return {"text": texts}


In [ ]:
# ── Attach the Llama-3.1 chat template ──────────────────────────────────────
# The base repo's tokenizer has no chat_template set, so apply_chat_template would
# raise 'chat_template is not set'. get_chat_template installs the correct Llama-3.1
# template (the <|start_header_id|>…<|eot_id|> formatting).
# MUST run this BEFORE the .map() that calls apply_chat_template.

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",   # OpenBioLLM is Llama-3.1-8B based
)

In [ ]:
# Verify the template attached (should print True).

print(tokenizer.chat_template is not None)   # should now be True

In [ ]:
# ── Apply the formatting to the whole dataset ───────────────────────────────
# batched=True processes rows in chunks for speed.
# remove_columns=... drops the original instruction/input/output, leaving only the
# single 'text' column SFTTrainer will train on.

dataset = dataset.map(format_chat_template, batched=True, remove_columns=dataset.column_names)

In [ ]:
# Extra shuffle after mapping (harmless; ensures order isn't correlated by source).

dataset = dataset.shuffle(seed=42)

In [ ]:
# How many formatted examples we have in total (~119k before subsampling).

len(dataset)

In [ ]:
# ── (Optional) push the formatted dataset to the Hub ────────────────────────
# Saves the ready-to-train 'text' dataset so you can reload it later without
# re-running the formatting. Requires Internet enabled + HF_TOKEN.
# NOTE: this is real (de-identified) patient data — consider a PRIVATE repo
# (private=True) given GDPR/sensitivity, and that automated de-id is imperfect.

dataset.push_to_hub("healthPilot-training-dataset", token=user_secrets.get_secret("HF_TOKEN"))

In [ ]:
# ── Train/eval split + SUBSAMPLE to 25k ─────────────────────────────────────
# train_test_split first carves off 2% as a held-out eval set (used to watch for
# overfitting via eval_loss). Then we SELECT 25,000 train rows.
# WHY 25k: the full ~117k takes 60+ hours on one T4 (won't fit a Kaggle session).
# 25k is plenty to learn the doctor voice/triage style for a Stage-1 prototype.
# IMPORTANT ORDER: split BEFORE select, so eval rows are never in the training pool.

split = dataset.train_test_split(test_size=0.02, seed=42)
train_ds, eval_ds = split["train"].select(range(25000)), split["test"]

print(f"train: {len(train_ds)}  |  eval: {len(eval_ds)}")

---
## 6 · Attach LoRA adapters
Add small trainable low-rank adapters (QLoRA) — we train these, not the full 8B.

In [ ]:
# The 7 transformer projection matrices LoRA will adapt (attention + MLP).
# (Defined here; the actual list passed to LoRA is in the next cell.)

modules = ["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"]

In [ ]:
# ── Attach LoRA adapters (Unsloth) ──────────────────────────────────────────
# We train small low-rank adapters, not the full 8B (that's the whole point of QLoRA).
# r=32 / alpha=32 : adapter capacity (Unsloth convention is alpha == r)
# lora_dropout=0  : Unsloth fast-path default
# target_modules  : the 7 matrices above (q/k/v/o + gate/up/down)
# use_gradient_checkpointing='unsloth' : the key VRAM saver that makes 8B fit on a T4
# CRITICAL: use FastLanguageModel.get_peft_model (NOT peft.get_peft_model) so the
#           Unsloth optimizations stay intact. Do NOT call this twice.

# LoRA via Unsloth
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                       # ↑ from 16: 8B + free-text generation benefits from more capacity
    lora_alpha=32,              # alpha == r (Unsloth's recommended default, not 2×r)
    lora_dropout=0,             # 0 is Unsloth-optimized (faster); your 0.05 is fine but unnecessary
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",   # big VRAM saver — key for 8B on T4
    random_state=42,
    use_rslora=False,
)

In [ ]:
# Tokenizer vocab size (sanity check; also catches accidental token additions).

len(tokenizer)

# Test Training

In [ ]:
# ── SMOKE TEST: 5 steps on 50 rows ──────────────────────────────────────────
# Purpose: prove the pipeline trains end-to-end (finite loss, no OOM) BEFORE the
# long run. report_to='none' keeps this throwaway run out of wandb.
# train_on_responses_only(...) masks loss to the assistant turn only — the markers
# below are the Llama-3 turn headers the masker keys off.
# ⚠️ BUG TO FIX: this references `train_dataset`, but the dataset variable in this
#   notebook is `train_ds` (from the split cell). Change `train_dataset.select(...)`
#   to `train_ds.select(range(50))` or this cell will NameError.
# (5-step loss bounces around ~3.0 — that's noise, not a trend. It only proves plumbing.)

# --- 2. Quick training smoke test: a few steps on a small slice ---
smoke_args = SFTConfig(
    output_dir="smoke_test",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    max_steps=5,                 # just 5 steps
    max_seq_length=1024,
    dataset_text_field="text",
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    report_to="none",            # don't log smoke test to wandb
)

smoke_trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset.select(range(50)),
    tokenizer=tokenizer,
    args=smoke_args,
    packing=False,
)
smoke_trainer = train_on_responses_only(
    smoke_trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
smoke_trainer.train()
print("Smoke test passed — loss is finite and steps ran without OOM.")

# Real Training

In [ ]:
# ── REAL training config (SFTConfig) ────────────────────────────────────────
# per_device_train_batch_size=8 : fills the T4 (raises GPU utilization; ~3,100 steps for 25k)
# num_train_epochs=1            : correct for a large dataset (more → overfit/voice loss)
# max_seq_length=1024           : long enough for full doctor answers (512 would truncate them)
# learning_rate=2e-4 + linear + warmup_steps=100 : standard LoRA schedule
# optim='adamw_8bit'            : memory-light optimizer (matters at 8B)
#
# CHECKPOINTING → HUB (so a Kaggle disconnect doesn't lose hours):
#   save_strategy='steps', save_steps=500   : write a checkpoint every 500 steps
#   push_to_hub=True, hub_strategy='all_checkpoints' : push FULL checkpoints
#       (optimizer + scheduler + trainer_state) to the Hub, enabling true resume
#   save_total_limit=2          : keep disk from filling
#
# EVAL (watch for overfitting):
#   eval_strategy='steps', eval_steps=500
#   load_best_model_at_end=True + metric_for_best_model='eval_loss' (lower is better)
#     → automatically restores the lowest-eval-loss checkpoint at the end
#
# ⚠️ NOTE on resume: 'all_checkpoints' only pushes what gets SAVED — if a run dies
#   BEFORE the first save_steps (500), no resumable checkpoint exists yet. Consider
#   save_steps=100 early on so a checkpoint lands sooner.

training_args = SFTConfig(
    output_dir=new_model,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    optim="adamw_8bit",
    num_train_epochs=1,
    max_seq_length=1024,
    dataset_text_field="text",
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=100,
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_total_limit=2,              # keep only the last 2 on disk (avoids filling Kaggle disk)
    push_to_hub=True,                # <-- pushes each checkpoint to the Hub
    hub_model_id="Victorano/healthpilot-openbiollm",  # the repo to push to
    hub_strategy="all_checkpoints",  # push every checkpoint, not just the final
    hub_token=user_secrets.get_secret("HF_TOKEN"),

    # ---- eval settings ----
    eval_strategy="steps",          # evaluate periodically during training
    eval_steps=500,                 # run eval every 200 steps
    per_device_eval_batch_size=2,
    save_strategy="steps",
    save_steps=500,                 # keep aligned with eval_steps
    load_best_model_at_end=True,    # restore the lowest-eval-loss checkpoint at the end
    metric_for_best_model="eval_loss",
    greater_is_better=False,        # lower eval loss = better

    seed=42,
    report_to="wandb",
)

In [ ]:
# Tokenizer length re-check before building the trainer.

len(tokenizer)

In [ ]:
# ── Build the trainer ───────────────────────────────────────────────────────
# Uses train_ds (the 25k subsample) and eval_ds (held-out 2%).
# ⚠️ STALE-CONFIG TRAP: if you edit the SFTConfig above, you MUST re-run THIS cell
#   to rebuild the trainer — editing the config alone does nothing to an already-built
#   trainer object. (This bit us before: batch size 'changed' but steps didn't.)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,         # note: train_ds now, not train_dataset
    eval_dataset=eval_ds,           # <-- the new bit
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

In [ ]:
# ── Response-only loss masking ──────────────────────────────────────────────
# Re-applies the masking to the REAL trainer (must be done after building it).
# Ensures loss is computed ONLY on the assistant's answer, not the long system
# prompt or the patient's message — sharpens generation quality noticeably.

# ---- response-only loss masking (train only on the assistant's answer) ----
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

# Training

In [ ]:
# ── Pre-flight verification (do this BEFORE train()) ─────────────────────────
# Confirms the config actually took effect on the LIVE trainer:
#   per_device_train_batch_size → should print 8
#   len(get_train_dataloader()) → should be ~3,125 (25000/8), NOT ~14,630
# If you see 14,630 here, the 25k subsample didn't apply — fix before training.

print(trainer.args.per_device_train_batch_size)   # should print 8
print(len(trainer.get_train_dataloader()))        # should be ~ (rows / 8), not 14630

In [ ]:
# ── Launch training ─────────────────────────────────────────────────────────
# Runs the full fine-tune. With checkpoint-to-Hub set, an overnight 'Save & Run All'
# commit survives idle-timeouts; resume from the last Hub checkpoint if it dies.
# Watch eval_loss in wandb: when it flattens/rises while train loss keeps dropping,
# that's the overfitting point (and load_best_model_at_end rolls back to the best).

trainer.train()

---
## 8 · Save / export the model
Three independent options — pick by serving target: **A** adapter-only (smallest),
**B** merged fp16 (easiest to serve with vLLM/TGI), **C** GGUF (llama.cpp/Ollama).

# Save Model

In [ ]:
# ── SAVE OPTION A: LoRA adapter only (~100–200MB) ───────────────────────────
# Saves just the trained adapter + tokenizer (base model loaded separately at serve
# time). Smallest artifact. Good if your serving stack does base + adapter.

# ---- A. Save the LoRA ADAPTER only (small, ~100-200MB) ----
# Best if you'll load base model + adapter at serve time.
model.save_pretrained(new_model)                       # local
tokenizer.save_pretrained(new_model)
model.push_to_hub(new_model,
                  token=user_secrets.get_secret("HF_TOKEN"))
tokenizer.push_to_hub(new_model,
                      token=user_secrets.get_secret("HF_TOKEN"))

In [ ]:
# ── SAVE OPTION B: merged 16-bit standalone model ───────────────────────────
# Merges the LoRA into the base and saves one self-contained fp16 model — easiest to
# serve with vLLM/TGI/transformers. ⚠️ The merge briefly needs ~16GB to hold the full
# 8B in fp16; may OOM on a T4. If so, save the adapter (A) and merge in a bigger session.

# ---- B. Merge LoRA into base, save as 16-bit (full standalone model) ----
# Best for serving with vLLM / TGI / transformers — one self-contained model.
model.save_pretrained_merged(
    f"{new_model}-merged-16bit",
    tokenizer,
    save_method="merged_16bit",
)
model.push_to_hub_merged(
    f"{new_model}-merged-16bit",
    tokenizer,
    save_method="merged_16bit",
    token=user_secrets.get_secret("HF_TOKEN"),
)

In [ ]:
# ── SAVE OPTION C: GGUF export (llama.cpp / Ollama) ─────────────────────────
# Converts to quantized GGUF for CPU/edge serving. q4_k_m = good size/quality balance.
# Note: GGUF is for llama.cpp-style runtimes — it does NOT let you run inside tiny
# serverless functions (still needs a process with the multi-GB file loaded).

# ---- C. Export to GGUF for llama.cpp / Ollama (quantized, CPU/edge serving) ----
model.push_to_hub_gguf(
    f"{new_model}-gguf",
    tokenizer,
    quantization_method="q4_k_m",   # good size/quality balance; also "q5_k_m", "q8_0", "f16"
    token=user_secrets.get_secret("HF_TOKEN"),
)

---
## 9 · Test the fine-tuned model
Run an emergency case and read the output critically (see the inline warnings).

# Test Fine Tuned Model

In [ ]:
# ── Test the fine-tuned model ───────────────────────────────────────────────
# Loads base+adapter, switches to inference mode, re-attaches the chat template,
# and runs one emergency case (classic heart-attack presentation).
#
# ⚠️ KNOWN ISSUES we found testing this:
#  1) eos_token_id=[..., 128009]: 128009 is the Llama-3 <|eot_id|>. Correct for
#     OpenBioLLM (Llama-3 based), but if you ever swap to a Gemma model this ID is
#     WRONG (Gemma uses <end_of_turn>). Keep it ONLY for the Llama-family model.
#  2) max_new_tokens=400 can truncate long answers — raise to ~1024.
#  3) CLINICAL: the fine-tuned adapter MIS-TRIAGED this heart-attack case (suggested
#     a stress test instead of 'call 999'), while the BASE model triaged it correctly.
#     → Treat the LLM's severity as a soft signal only; the 999/booking escalation
#       MUST be a deterministic rule layer reading symptoms/nodes, biased to over-triage.

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Victorano/healthpilot-openbiollm", max_seq_length=1024, load_in_4bit=True)
FastLanguageModel.for_inference(model)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",   # OpenBioLLM is Llama-3.1-8B based
)

messages = [
    {"role": "system", "content": SYSTEM_INSTRUCTION},
    {"role": "user", "content": "I'm a 45-year-old man with crushing chest pain spreading to my left arm and I feel sweaty and short of breath."},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=400,
                     eos_token_id=[tokenizer.eos_token_id, 128009])  # 128009 = <|eot_id|>
print(tokenizer.decode(out[0], skip_special_tokens=True))